# In-class Session: Train and Test ML/DL Models
AI for Materials Science — Hands-on session 5

Today we train and test machine-learning models that predict the **average voltage** of lithium
insertion electrodes from their composition. Sections A to D are worked examples: a linear baseline,
tree ensembles with cross-validation, and a neural network in PyTorch. In section E you train one
model yourself; that is what you submit for attendance.

**Do this first:** 1. File ▸ Save a copy in Drive. 2. In your copy, run 0-1 and 0-2 right away; they
install while we review. If you ran 0-1 before saving, run 0-1 and 0-2 again in your copy now.

## Today's plan

### 0 · Setup
Install two libraries, download the course data, import everything.

### A · The data and its features
Load the electrode table and turn each formula into the 145 composition features from session 2.

### B · Split, a linear baseline, three metrics
Split by composition, fit a linear model, read MAE, RMSE and R², and look at a parity plot. See how much the test score depends on the split.

### C · Random forest, gradient boosting, cross-validation
Compare three models with 5-fold cross-validation, see why one tree is not enough, see which features
matter, and evaluate a yes/no classification with precision, recall and F1.

### D · A neural network in PyTorch
Build an MLP, watch the train and validation loss, compare it with the trees, and measure how much of the test error comes from identical compositions.

### E · Assignment (attendance)
Pick one model (or the other target), train and evaluate it, and save a results table and a parity plot.

The sections follow one workflow that you can reuse for any property: turn every material into a row of
numbers (A), set a test set aside and fit a simple baseline (B), compare stronger models by cross-validation
on the training data only (C, D), and use the test set only to report scores (B-4, C-3, C-5, D-7, E), never to
choose a model. Each step starts with a short note on what the code does and what to look for in its output.

---

Data: Materials Project insertion electrodes (Li). Jain et al., APL Mater. 1, 011002 (2013); accessed 2026-10-22.

Run the cells one at a time from the top. Later cells reuse variables created in earlier ones, so skipping ahead gives you a "name is not defined" error. Lines starting with `##` inside the code are comments written for you; Python does not run them.

## 0. Setup

### 0-1. Install the libraries
Only `matminer` and `pymatgen` need installing; scikit-learn, PyTorch, pandas and Matplotlib ship with Colab.
The versions are pinned to the ones this notebook was tested with. The install takes a minute or two.
`-q` keeps pip quiet. pip may still print warnings about other packages in Colab; carry on once the cell has
finished. If the two libraries cannot be imported after all, 0-3 says so and A-3 reads the saved features.

In [ ]:
!pip install -q matminer==0.10.1 pymatgen==2026.9.24

### 0-2. Download the course material
The same repository as in the earlier sessions. `Data/` holds the electrode table and a saved copy of its
features. If git says the destination path already exists, the folder is already there: carry on.

In [ ]:
!git clone https://github.com/kwongibaek/MS49900-AI4M.git

### 0-3. Import the libraries
The seeds make the random parts (splits, forests, network initialisation) repeatable. The `try` block
imports pymatgen and matminer only if 0-1 installed them, and `HAVE_MATMINER` records which way A-3 will
go. The `<torch._C.Generator ...>` line under the cell comes from `torch.manual_seed` and is harmless.

In [ ]:
import copy, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import (HistGradientBoostingRegressor, RandomForestClassifier,
                              RandomForestRegressor)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, f1_score,
                             mean_absolute_error, mean_squared_error, precision_score,
                             r2_score, recall_score)
from sklearn.model_selection import (GroupKFold, GroupShuffleSplit, cross_val_predict,
                                     cross_validate)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor, plot_tree
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
try:   ## only the live featurizer in A-3 needs these two libraries
    from pymatgen.core import Composition
    from matminer.featurizers.base import MultipleFeaturizer
    from matminer.featurizers.composition import (ElementProperty, IonProperty,
                                                  Stoichiometry, ValenceOrbital)
    HAVE_MATMINER = True
except ImportError as err:   ## 0-1 failed: A-3 reads the saved features instead
    HAVE_MATMINER = False
    print("matminer/pymatgen not importable; A-3 uses the saved features.", err)

np.random.seed(0)
torch.manual_seed(0)

### 0-4. Set the file paths
`OUTPUT` is the folder for today's figures and for the two files you submit in E. `True` means 0-2 worked;
if it prints `False`, run 0-2 again.

In [ ]:
DATA_DIR = Path("MS49900-AI4M/Data")
OUTPUT = Path("outputs")
OUTPUT.mkdir(exist_ok=True)
print("Data folder exists:", DATA_DIR.exists())

## A. The data and its features

### A-1. Load the electrode table
One row is one lithium insertion electrode computed by the Materials Project: the host framework, the
formula at full discharge, and the average voltage against Li/Li⁺ along the lithiation path.

`battery_formula` shows the lithium range: `Li0-2FePO4F` is a host that takes between 0 and 2 Li per formula
unit, and `formula_discharge` is its fully lithiated end, `Li2FePO4F`.

In [ ]:
df = pd.read_csv(DATA_DIR / "mp_li_electrodes.csv")
print("rows, columns:", df.shape)
cols = ["battery_formula", "formula_discharge",
        "average_voltage", "capacity_grav"]
df[cols].head()

### A-2. What one row means
`formula_discharge` is the fully lithiated composition; `average_voltage` (V vs Li/Li⁺) is today's target;
`capacity_grav` (mAh/g) is the alternative target for the assignment. The voltage comes from DFT reaction energies
against Li metal; the capacity from how much Li the host takes up per gram. Both are computed, not measured.

`describe()` summarises each column: count, mean, standard deviation (`std`), minimum, quartiles and
maximum. Keep the `std` of `average_voltage` in mind: a model that predicts the mean voltage for every
electrode has an RMSE of about that size, so a useful model has to do clearly better. `capacity_grav` has
a long tail: its maximum is far above the `75%` row.

In [ ]:
df[["average_voltage", "capacity_grav", "nelements"]].describe().round(2)

### A-3. From formula to 145 composition features
The same featurizer as in session 2: stoichiometry (6), Magpie element-property statistics (132),
valence-orbital fractions (4) and ionic attributes (3). If matminer could not be installed, or the computation
fails in Colab, the cell reads a saved copy of exactly the same features (computed by the TA with the same
pinned versions), so everyone continues with the same matrix.

Why features at all: a model needs a row of numbers of fixed length for every sample, and a formula string
is not that. Each formula becomes 145 numbers that describe its elements and their proportions; for example
`MagpieData mean Electronegativity` is the composition-weighted mean electronegativity of its elements.
With `ignore_errors=True`, a formula the featurizer cannot handle gets empty (NaN) values instead of
stopping the cell. Today's table has none, and the imputers in B-3 and C-1 would fill them.

In [ ]:
X_all, source = None, "saved copy (matminer not installed)"
if HAVE_MATMINER:
    try:   ## construction and computation are both optional
        featurizer = MultipleFeaturizer([
            Stoichiometry(), ElementProperty.from_preset("magpie"),
            ValenceOrbital(props=["frac"]), IonProperty(fast=True)])
        featurizer.set_n_jobs(1)
        comps = [Composition(f) for f in df["formula_discharge"]]
        feats = featurizer.featurize_many(comps, ignore_errors=True,
                                          pbar=False)
        X_all = pd.DataFrame(feats, index=df.index,
                             columns=featurizer.feature_labels())
        source = "computed now"
    except Exception as err:   ## either failed: use the saved copy below
        source = f"saved copy ({type(err).__name__})"
if X_all is None:   ## the saved copy of exactly the same 145 features
    saved = pd.read_csv(DATA_DIR / "mp_li_electrodes_magpie145.csv.gz")
    X_all = df[["battery_id"]].merge(saved, on="battery_id", how="left")
    X_all = X_all.drop(columns="battery_id")
FEATURE_NAMES = list(X_all.columns)
print("feature matrix:", X_all.shape, "from", source)

## B. Split, a linear baseline, three metrics

### B-1. Repeated compositions
The same discharged formula can appear in several rows (different host structures). If such rows land
in both train and test, the model is partly tested on data it has already seen.

`duplicated(keep=False)` marks every row whose formula occurs more than once. A random row split would put
some copies in training and others in the test set; the model could then do well partly by recognising a
formula it has already seen, and the test score would overestimate how well the model does on a new composition.

In [ ]:
repeated = int(df["formula_discharge"].duplicated(keep=False).sum())
unique = df["formula_discharge"].nunique()
print(f"{repeated} rows share their formula with another row",
      f"({unique} unique formulas in {len(df)} rows)")

### B-2. Split by composition
`GroupShuffleSplit` keeps every row of one formula on the same side. 80% of the formulas go to the training set and 20% to the test set.

Each formula is one **group**. `random_state=42` fixes which formulas go to the test set, so everyone works
with the same split. The `assert` stops the notebook if a formula ended up on both sides. `groups_train` is
kept for the cross-validation in C.

In [ ]:
groups = df["formula_discharge"]
y_all = df["average_voltage"]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X_all, y_all, groups=groups))
X_train, X_test = X_all.iloc[train_idx], X_all.iloc[test_idx]
y_train, y_test = y_all.iloc[train_idx], y_all.iloc[test_idx]
groups_train = groups.iloc[train_idx]
leaked = set(groups_train) & set(groups.iloc[test_idx])
assert not leaked, "a formula leaked into both sets"
print("train rows:", len(X_train), " test rows:", len(X_test))

### B-3. A linear baseline
A pipeline fills missing feature values with the median, standardises every column, and fits ordinary
least squares. The imputer and scaler are fitted on the training rows only.

A **pipeline** chains the steps: `fit` fits them in order on the training rows, and `predict` applies the
same fitted steps to any new rows. That is what keeps the test rows out of the medians and the scaling
statistics; computing those on all rows would leak information from the test set into training. For plain
least squares the scaling does not change the predictions, but it puts all coefficients on one scale, and
the same pipeline then works for models that need it. The output is a summary of the pipeline.

In [ ]:
linear = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                       LinearRegression())
linear.fit(X_train, y_train)

### B-4. Three metrics
MAE and RMSE are in volts; R² is dimensionless; the formulas are written out in preclass A-4. Read the train and test columns side by side.

`metrics` works from true and predicted values, `scores` from a model and a data set; C, D and E reuse
both. Two questions to ask of the table: how much worse is the test column than the train column, and how
does the test RMSE compare with the voltage `std` from A-2? R² answers the second one in relative terms: R² = 1 means every prediction is exact, R² = 0 means no better than
predicting the mean, and on test data it can even be negative.

In [ ]:
def metrics(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2_score(y_true, y_pred)}

def scores(model, X, y):
    return metrics(y, model.predict(X))

pd.DataFrame({"train": scores(linear, X_train, y_train),
              "test": scores(linear, X_test, y_test)}).round(3)

### B-5. Parity plot
True voltage on the x-axis, predicted on the y-axis, for the train and test rows.

Points above the dashed diagonal are over-predictions, points below it under-predictions. Look at the two
ends of the voltage range: if the cloud is flatter than the diagonal, the model pulls high and low voltages
toward the mean, a common sign of a model that is too simple. `parity` is a function so that D-7 and E can
reuse it.

Look also at the lowest test predictions: a few electrodes get voltages far outside anything in the data, on this
split down to about −6 V. Many of the 145 features are near-copies of each other, and least squares can give such
features large coefficients of opposite sign that cancel on the training rows. For a composition unlike those rows
they no longer cancel, and the prediction becomes extreme.

In [ ]:
def parity(ax, y_true, y_pred, title, target="average voltage", unit="V"):
    ax.scatter(y_true, y_pred, s=10, alpha=0.6)
    lo = float(min(y_true.min(), y_pred.min()))
    hi = float(max(y_true.max(), y_pred.max()))
    ax.plot([lo, hi], [lo, hi], "k--", linewidth=1)
    ax.set_xlabel(f"true {target} ({unit})")
    ax.set_ylabel(f"predicted ({unit})")
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
parity(axes[0], y_train, linear.predict(X_train), "Linear - train")
parity(axes[1], y_test, linear.predict(X_test), "Linear - test")
fig.tight_layout()
fig.savefig(OUTPUT / "B5_parity_linear.png", dpi=150, bbox_inches="tight")
plt.show()

### B-6. How much does the test score depend on the split?
The test score in B-4 comes from one random split of the formulas (`random_state=42`). This cell refits the
same linear pipeline on ten other splits by composition and scores each on its own held-out 20%. It only shows
how much a single hold-out score moves; it chooses nothing, and the rest of the notebook keeps the split from
B-2.

Compare the range with the voltage `std` of about 1.1 V from A-2: on an unlucky split the linear model is worse
than predicting the mean, on a lucky one it looks noticeably better than in B-4. With a few thousand rows one
hold-out score is a noisy estimate. That is why C-2 compares models with five folds and reports their spread:
with limited data, cross-validation is the more reliable estimate.

Why the linear model swings so much: on the worst splits, the extreme predictions seen in B-5 fall in the test set.
Dozens of feature pairs correlate above 0.95, and plain least squares has nothing that keeps its coefficients
small. Regularisation, one of the lecture's remedies for overfitting, does exactly that: a ridge penalty, for
example, brings the worst of these ten splits from about 1.8 V down to about 1.1 V. You will try regularisation on
the neural network in Practice 2.

In [ ]:
holdout_rmse = []
for seed in range(10):   ## ten other 80/20 splits by composition
    split_s = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr, te = next(split_s.split(X_all, y_all, groups=groups))
    lin_s = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                          LinearRegression())
    lin_s.fit(X_all.iloc[tr], y_all.iloc[tr])
    holdout_rmse.append(scores(lin_s, X_all.iloc[te], y_all.iloc[te])["RMSE"])
print("test RMSE on ten splits (V):", np.round(holdout_rmse, 3))
print(f"from {min(holdout_rmse):.3f} to {max(holdout_rmse):.3f} V;",
      f"the split in B-4: {scores(linear, X_test, y_test)['RMSE']:.3f} V")

## C. Random forest, gradient boosting, cross-validation

### C-1. Three models
The linear model again, a random forest of 200 trees, and histogram gradient boosting (which handles
missing values itself, so it needs no imputer).

- **Random forest**: each of the 200 trees is grown on a bootstrap sample of the training rows (rows drawn
  with replacement), and the forest predicts the average of its trees. One deep tree follows its training
  rows too closely; the average of many slightly different trees is much steadier.
- **Gradient boosting**: trees are added one after another, and each new, small tree is fitted to the errors
  that the trees before it still make. "Histogram" means every feature is first sorted into at most 255
  bins, which makes the search for good splits fast.

A tree splits on one feature at a time at a threshold, so rescaling a feature changes nothing: neither tree
model has a `StandardScaler`. `cross_validate` reports scores where higher is better, so errors come back
negative (`neg_...`); `cv_scores` flips the sign and returns the mean over the folds plus the spread (std)
of the RMSE. This cell only defines the models; nothing is trained yet.

In [ ]:
models = {
    "Linear": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                            LinearRegression()),
    "RandomForest": make_pipeline(SimpleImputer(strategy="median"),
                                  RandomForestRegressor(n_estimators=200,
                                                        n_jobs=-1, random_state=0)),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=0),
}
scoring = {"MAE": "neg_mean_absolute_error",
           "RMSE": "neg_root_mean_squared_error", "R2": "r2"}

def cv_scores(model, cv):
    res = cross_validate(model, X_train, y_train, groups=groups_train,
                         cv=cv, scoring=scoring)
    return {"CV MAE": -res["test_MAE"].mean(), "CV RMSE": -res["test_RMSE"].mean(),
            "CV RMSE std": res["test_RMSE"].std(), "CV R2": res["test_R2"].mean()}

### C-2. Five-fold cross-validation on the training set
`GroupKFold` keeps formulas together inside each fold too. Each model is fitted five times; we report the
mean and spread of the validation RMSE. The test set is not touched.

The training formulas are divided into five folds. Each model is trained on four folds and validated on the
fifth, five times over, so every training row is validated once by a model that did not see it. The mean
is the estimate; the std shows how much it depends on which formulas fall into a fold, and a difference
between two models smaller than their std is not convincing. Why not simply compare on the test set? Once
you choose by the test score, the test score is no longer an honest estimate for new materials. The cell
trains 15 models and can take up to a minute.

In [ ]:
cv = GroupKFold(n_splits=5)
rows = [{"model": name, **cv_scores(model, cv)}
        for name, model in models.items()]
cv_table = pd.DataFrame(rows).set_index("model").round(3)
cv_table

### C-2b. One tree versus a forest
The random forest in C-2 averages 200 decision trees. Here is one tree on its own. A tree asks one yes/no
question about one feature at each node: the first question is the **root**, the ones below it are **interior
nodes**, and every **leaf** predicts the mean voltage of the training rows that end up there. The first cell
draws a tree only two levels deep so that you can read it: `value` is the leaf's prediction in volts and
`samples` the number of training rows that reach it.

The second cell scores trees of three depths with the same grouped folds as C-2 and puts the training RMSE next
to the CV RMSE. The deeper the tree, the closer it follows its training rows; without a depth limit it almost
reproduces them, yet on held-out formulas it does worse than a shallow tree. That is the overfitting from the
lecture. Compare the CV RMSE with the forest's in C-2: averaging many deep trees, each grown on different rows,
removes most of it.

In [ ]:
small_tree = make_pipeline(SimpleImputer(strategy="median"),
                           DecisionTreeRegressor(max_depth=2, random_state=0))
small_tree.fit(X_train, y_train)
fig, ax = plt.subplots(figsize=(12, 4.5))
plot_tree(small_tree[-1], ax=ax, filled=True, impurity=False, precision=2,
          feature_names=[n.replace("MagpieData ", "") for n in FEATURE_NAMES])
plt.show()

In [ ]:
tree_rows = []
for depth in [3, 6, None]:   ## None: no depth limit
    tree = make_pipeline(SimpleImputer(strategy="median"),
                         DecisionTreeRegressor(max_depth=depth, random_state=0))
    res = cross_validate(tree, X_train, y_train, groups=groups_train, cv=cv,
                         scoring="neg_root_mean_squared_error", return_train_score=True)
    tree_rows.append({"max_depth": str(depth), "train RMSE": -res["train_score"].mean(),
                      "CV RMSE": -res["test_score"].mean()})
pd.DataFrame(tree_rows).set_index("max_depth").round(3)

### C-3. Fit on train, score on test
Now each model is fitted once on all training rows and scored on the held-out test rows.

For the two tree models this is the first look at the test rows. Compare the test RMSE here with the CV
RMSE in C-2: when the two are close, cross-validation was a good guide. `fit s` is the training time in seconds, a cost to weigh alongside accuracy. `fitted` keeps the trained models for C-4 and D-7.

In [ ]:
fitted, test_rows = {}, []
for name, model in models.items():
    t0 = time.time()
    fitted[name] = model.fit(X_train, y_train)
    fit_s = time.time() - t0   ## training only, before predicting the test rows
    test_rows.append({"model": name, **scores(fitted[name], X_test, y_test),
                      "fit s": fit_s})
test_table = pd.DataFrame(test_rows).set_index("model").round(3)
test_table

### C-4. Which features matter?
A random forest records how much each feature reduced the error across its trees. The top 15 are shown.

This is the **mean decrease in impurity (MDI)**: whenever a tree splits on a feature, the drop in squared
error at that split is credited to the feature; the credits are summed over all trees and scaled so that
all features add up to 1. Two cautions: it is measured on the training data, and features that carry the
same information (many Magpie statistics are correlated) share the credit, so a short bar does not prove
that a feature is useless. Names that start with `MagpieData` are one statistic (mean, range, mode, ...) of
one element property over the composition.

In [ ]:
rf = fitted["RandomForest"].named_steps["randomforestregressor"]
importance = pd.Series(rf.feature_importances_, index=FEATURE_NAMES)
importance = importance.sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
importance.head(15)[::-1].plot.barh(ax=ax)
ax.set_xlabel("importance"); ax.set_title("Random forest - top 15 features")
fig.tight_layout()
fig.savefig(OUTPUT / "C4_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

**Why is covalent radius at the top?** The voltage is set by electronic structure: the energy of the states in
the host that the inserted lithium's electron fills. The model never sees that; it sees only statistics of the
composition, so it relies on whichever statistic is the best proxy for it. Oxygen (66 pm) and fluorine (57 pm) have
far smaller covalent radii than lithium, the transition metals and alloying elements such as Sn or Bi (about
120–160 pm). `mean CovalentRadius` is therefore low for formulas rich in O and F and high for sulfides and alloys
without an anion. The anion is one of the main factors that set the redox energy
(Aydinol et al., *Phys. Rev. B* 56, 1354 (1997), trace the voltage gap between oxides, sulfides and selenides to
charge transfer to the anion), and the cell below shows how strongly the anion type separates the voltages in this table.

So covalent radius is a proxy for that chemistry, not its cause. And because correlated features share the
credit, which of several such proxies ranks first is partly arbitrary.

In [ ]:
## The elements column lists each electrode's elements other than Li, for example "Fe-P-O-F".
elements = df["elements"].str.split("-")
anion = np.select(
    [elements.apply(lambda e: "F" in e), elements.apply(lambda e: "O" in e),
     elements.apply(lambda e: bool({"S", "Se", "Te"} & set(e)))],
    ["contains F", "contains O (no F)", "S, Se or Te (no O, F)"], default="none of these")
anion_table = pd.DataFrame({"anion": anion, "voltage": df["average_voltage"],
                            "radius": X_all["MagpieData mean CovalentRadius"]})
(anion_table.groupby("anion")
 .agg(rows=("voltage", "size"), mean_voltage=("voltage", "mean"),
      mean_covalent_radius=("radius", "mean"))
 .sort_values("mean_voltage", ascending=False).round(2))

### C-5. A yes/no question: classification metrics
Screening often needs a yes/no answer rather than a number: is this electrode a **high-voltage cathode
candidate** (average voltage of at least 4.5 V)? About one electrode in five is. The features and the split by
composition stay the same; the target becomes 1 (yes) or 0 (no), and a `RandomForestClassifier` returns the
probability of yes. We answer yes when that probability reaches a **cut-off**, 0.5 by default.

The metrics from the lecture, with TP, FP, FN and TN counted in a **confusion matrix**:
- **Accuracy**: the share of correct answers. With one yes in five, answering no every time already scores about
  0.8 and finds no candidate at all; that is the first row of the table.
- **Precision** = TP / (TP + FP): of the electrodes we call yes, the share that really are.
- **Recall** = TP / (TP + FN): of the real high-voltage electrodes, the share we find.
- **F1**: the harmonic mean of precision and recall; it is high only when both are.

For screening, a missed candidate is lost for good while a false alarm costs one extra DFT check, so recall
must not be too low; a list full of false alarms is no use either. Lowering the cut-off trades precision for
recall. The cut-off is a setting like any other, so the first cell chooses it by the best F1 on out-of-fold
predictions of the training rows (`cross_val_predict` with the folds from C-2); the test rows are used only for the report in the second cell. The two cells take about 15 seconds.

Note for comparison with the lecture slides: scikit-learn puts the **true** class in the rows of the
confusion matrix and the prediction in the columns; the slides put the prediction in the rows. Read the axis
labels.

In [ ]:
CUT_V = 4.5   ## yes = average voltage of at least 4.5 V
c_train = (y_train >= CUT_V).astype(int)
c_test = (y_test >= CUT_V).astype(int)
print(f"share of yes: train {c_train.mean():.3f}, test {c_test.mean():.3f}")

clf = make_pipeline(SimpleImputer(strategy="median"),
                    RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0))
## Out-of-fold probabilities: each training row is predicted by a model that did not see it.
p_oof = cross_val_predict(clf, X_train, c_train, groups=groups_train, cv=cv,
                          method="predict_proba")[:, 1]
cutoffs = np.round(np.arange(0.1, 0.91, 0.05), 2)
cv_f1 = [f1_score(c_train, p_oof >= t, zero_division=0) for t in cutoffs]
best_cutoff = float(cutoffs[int(np.argmax(cv_f1))])
print(f"CV F1 at cut-off 0.5: {cv_f1[list(cutoffs).index(0.5)]:.3f};",
      f"best CV F1 {max(cv_f1):.3f} at cut-off {best_cutoff}")

In [ ]:
def class_metrics(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred),
            "F1": f1_score(y_true, y_pred, zero_division=0)}

clf.fit(X_train, c_train)
p_test = clf.predict_proba(X_test)[:, 1]
answers = {"always no": np.zeros(len(c_test), dtype=int),
           "forest, cut-off 0.5": (p_test >= 0.5).astype(int),
           f"forest, CV cut-off {best_cutoff}": (p_test >= best_cutoff).astype(int)}
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, name in zip(axes, list(answers)[1:]):
    ConfusionMatrixDisplay.from_predictions(c_test, answers[name], ax=ax, colorbar=False,
                                            display_labels=["below 4.5 V", "4.5 V or more"])
    ax.set_title(name)
fig.tight_layout()
fig.savefig(OUTPUT / "C5_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
pd.DataFrame({name: class_metrics(c_test, pred) for name, pred in answers.items()}).T.round(3)

### C-6. Practice 1
Change **one** hyperparameter of the random forest (for example `n_estimators`, `max_depth` or
`max_features`) and run the 5-fold cross-validation again. Report the CV RMSE before and after, and write
one sentence on what changed and why.

What the three hyperparameters do: `n_estimators` is the number of trees (more trees give a steadier average and take
longer); `max_depth` limits how deep each tree grows (shallower trees are simpler); `max_features` is the
share of the features a split may choose from (smaller values make the trees more different from each
other). Keep `random_state=0`, so that your change is the only difference.

Write it yourself in the empty cell below. Reuse `cv`, `groups_train` and `cv_scores` from C-1 and C-2.

In [ ]:
## Write your code here.
## Order: copy the RandomForest pipeline with one changed parameter -> cv_scores(...) -> print both CV RMSEs -> one sentence

## D. A neural network in PyTorch

### D-1. A validation split
We hold out 20% of the **training** formulas as a validation set: it tells us how the network is doing
after every epoch. No decision about the network uses the test rows; the MLP's test score appears only in
D-7. The split comes first, so that the scaling in D-2 is fitted on the fit rows only; the validation rows
must not influence it.

From here on there are three sets: the **fit** rows update the weights, the **validation** rows monitor the training and select the checkpoint, and the **test** rows score the network once, in D-7. This split,
too, keeps every formula on one side.

In [ ]:
val_split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
fit_idx, val_idx = next(val_split.split(X_train, y_train, groups=groups_train))
X_fit, X_val = X_train.iloc[fit_idx], X_train.iloc[val_idx]
y_fit = y_train.iloc[fit_idx].to_numpy(dtype=np.float32)
y_val = y_train.iloc[val_idx].to_numpy(dtype=np.float32)
print("fit rows:", len(X_fit), " validation rows:", len(X_val))

### D-2. Scale the inputs and build data loaders
Neural networks train badly on features with very different ranges. We impute and standardise with
statistics from the fit rows only, convert everything to float32 tensors, and let a `DataLoader` serve
mini-batches of 64 rows.

With mini-batches the weights are updated after every 64 rows instead of once per pass, so one epoch makes
many small steps; `shuffle=True` reshuffles the rows into new batches every epoch. `reshape(-1, 1)` turns the targets
into a column, the same (rows, 1) shape as the network's output. `X_test_s` is the test set scaled with the
same fit-row statistics, ready for D-7.

In [ ]:
prep = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
prep.fit(X_fit)

def to_tensor(a):
    return torch.tensor(np.asarray(a, dtype=np.float32))

X_fit_t = to_tensor(prep.transform(X_fit))
y_fit_t = to_tensor(y_fit).reshape(-1, 1)
X_val_t = to_tensor(prep.transform(X_val))
y_val_t = to_tensor(y_val).reshape(-1, 1)
X_test_s = prep.transform(X_test)
y_test_a = y_test.to_numpy(dtype=np.float32)
train_loader = DataLoader(TensorDataset(X_fit_t, y_fit_t),
                          batch_size=64, shuffle=True)
print("scaled fit matrix:", tuple(X_fit_t.shape),
      " batches per epoch:", len(train_loader))

### D-3. Build an MLP
145 inputs → 128 → 64 → 1 output, with ReLU between the layers. Every connection between two layers has a trainable weight.

`nn.Linear(145, 128)` computes 128 weighted sums of the 145 inputs, each with its own bias. `ReLU` keeps
positive values and sets negative ones to zero. That nonlinearity is what lets the network follow curved
relationships: linear layers stacked without it collapse into one linear model. `Dropout(p)` sets a random
share p of the hidden values to zero during training; with the default p = 0 it does nothing (D-6 lets
you raise it). The parameter count is 145·128 + 128 + 128·64 + 64 + 64 + 1 = 27,009, far more than there
are fit rows: the network can memorise them, and D-5 shows when it starts to.

In [ ]:
def make_mlp(n_in, dropout=0.0):
    return nn.Sequential(nn.Linear(n_in, 128), nn.ReLU(), nn.Dropout(dropout),
                         nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))

torch.manual_seed(0)
mlp = make_mlp(X_fit_t.shape[1])
n_params = sum(p.numel() for p in mlp.parameters())
print(mlp)
print("trainable parameters:", n_params)

### D-4. The training loop
The same four steps as in the preclass loop, now over mini-batches. After every epoch both losses are
measured with the same weights in evaluation mode, on the fit rows and on the validation rows, so the two
curves are comparable. We remember the weights of the epoch with the lowest validation loss and restore
them at the end (a validation checkpoint); the loop itself runs for 200 epochs or 60 seconds, whichever
comes first. The first cell defines the function; the second calls it.

`Adam` adapts the step size of every weight from its recent gradients; `lr=1e-3` is its usual starting
value. `model.train()` and `model.eval()` switch dropout on for the updates and off for the measurements.
The checkpoint is a `copy.deepcopy` of the weights, because `state_dict()` returns the live tensors, which the next training steps keep changing.

In [ ]:
def train_mlp(model, loader, X_val, y_val, epochs=200, lr=1e-3,
              weight_decay=0.0, max_seconds=60):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    X_fit_all, y_fit_all = loader.dataset.tensors   ## the whole fit set, for the epoch-end loss
    history = {"train": [], "val": []}
    best = {"loss": float("inf"), "epoch": 0, "state": None}
    start = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()              ## 1. forget the previous gradients
            loss = loss_fn(model(xb), yb)      ## 2. how wrong are we?
            loss.backward()                    ## 3. gradients
            optimizer.step()                   ## 4. one step downhill
        model.eval()                           ## same weights, same mode for both losses
        with torch.no_grad():
            train_loss = loss_fn(model(X_fit_all), y_fit_all).item()
            val_loss = loss_fn(model(X_val), y_val).item()
        history["train"].append(train_loss)
        history["val"].append(val_loss)
        if val_loss < best["loss"]:
            best = {"loss": val_loss, "epoch": epoch,
                    "state": copy.deepcopy(model.state_dict())}
        if time.time() - start > max_seconds:
            break
    model.load_state_dict(best["state"])       ## restore the best validation checkpoint
    history.update(epochs_completed=epoch, seconds=time.time() - start,
                   best_epoch=best["epoch"])
    return history

In [ ]:
EPOCHS, MAX_SECONDS = 200, 60
history = train_mlp(mlp, train_loader, X_val_t, y_val_t,
                    epochs=EPOCHS, max_seconds=MAX_SECONDS)
epochs_completed = history["epochs_completed"]
seconds, best_epoch = history["seconds"], history["best_epoch"]
print(f"epochs completed {epochs_completed}/{EPOCHS}",
      f"in {seconds:.1f} s; best epoch {best_epoch}")

### D-5. Loss curves
Training loss keeps falling as long as you train; validation loss stops improving once the network starts
memorising the training rows, and may rise again. That turning point is where overfitting begins; the
dotted line marks the checkpoint we kept.

The y-axis is the mean squared error in V² on a log scale. Its square root is an RMSE in volts that you can
compare with the tables in B and C.

In [ ]:
def plot_history(h, title):
    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.plot(h["train"], label="train"); ax.plot(h["val"], label="validation")
    ax.axvline(h["best_epoch"] - 1, color="gray", linestyle=":",
               label=f"best epoch {h['best_epoch']}")
    ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("MSE loss (V²)")
    ax.set_title(title); ax.legend()
    return fig

fig = plot_history(history, "MLP - train vs validation loss")
fig.savefig(OUTPUT / "D5_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### D-6. Practice 2
Add **one** regulariser to the MLP: dropout (`make_mlp(n_in, dropout=0.2)`), weight decay
(`weight_decay=1e-4` in `train_mlp`), or a smaller training budget (fewer `epochs`). Train it with the same
loop, plot its curves with `plot_history`, and compare its best validation RMSE with the unregularised
network's. No test-based tuning: regularisers are chosen on validation data, and the MLP's test score
appears only in D-7.

What each one does: dropout makes the network work without relying on any single hidden unit; weight decay
pulls every weight a little toward zero at each step; fewer epochs stop the training before memorising sets
in. Call `torch.manual_seed(0)` before `make_mlp`, so that both networks start from the same initial weights
and the regulariser is the only difference. The best validation RMSE is the square root of the smallest
value in `history["val"]`.

Write it yourself in the empty cell below.

In [ ]:
## Write your code here.
## Order: torch.manual_seed(0) -> make_mlp(...) -> train_mlp(...) -> plot_history(...) -> best validation RMSE for both networks

### D-7. ML or DL? Four models on one test set
The three scikit-learn models from C-3 and the MLP, scored once on the same held-out rows. Every choice
so far was made with cross-validation or the validation set; this table is the final report, not a place
to pick a winner and tune again. Read the error columns and the fitting time together.

The second cell draws test parity plots for the tree model that won the cross-validation in C-2 and for the
MLP. On a few thousand rows of fixed features like these, tree ensembles are often hard to beat; neural
networks tend to pay off with much more data, or with inputs such as crystal structures from which they
learn their own features.

In [ ]:
mlp.eval()
with torch.no_grad():
    mlp_pred = mlp(to_tensor(X_test_s)).numpy().ravel()
four = test_table.copy()
four.loc["MLP (PyTorch)"] = {**metrics(y_test_a, mlp_pred),
                             "fit s": history["seconds"]}
four = four.round(3)
four

In [ ]:
trees = ["RandomForest", "HistGradientBoosting"]
best_tree = cv_table.loc[trees, "CV RMSE"].idxmin()   ## chosen by CV in C-2, not by test
fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
parity(axes[0], y_test, fitted[best_tree].predict(X_test), f"{best_tree} - test")
parity(axes[1], y_test, mlp_pred, "MLP - test")
fig.tight_layout()
fig.savefig(OUTPUT / "D7_parity_tree_vs_mlp.png", dpi=150, bbox_inches="tight")
plt.show()

### D-8. How much error do identical compositions cause?
Rows that share a formula have identical features (B-1) but can have different voltages, because their host
structures differ. A model that sees only the composition gives them all the same prediction, at best the mean
voltage of that formula. The cell computes this spread on the test rows: the lowest test RMSE that any
composition-only model could reach on these 552 rows, because it cannot tell rows with the same formula apart.

Read it as a property of today's data, not as a limit for new compositions. More than half of all rows have a
formula that appears only once, so their polymorph spread is never measured and counts as zero here; a material
with unseen polymorphs can be harder to predict than this number suggests.

Compare it with the best test RMSE in D-7. If the two were close, better models could not help on these rows and
only new inputs, such as the crystal structure, would. Here the gap is large, so most of today's test error is not
caused by identical compositions: more data, better features or better models can still pay off. The deep tree in
C-2b came down to about this level on its training rows, by memorising them.

In [ ]:
## Each test row minus the mean voltage of the test rows with the same formula.
groups_test = groups.iloc[test_idx]
spread = y_test - y_test.groupby(groups_test).transform("mean")
floor = float(np.sqrt(np.mean(spread ** 2)))
print(f"lowest test RMSE of any composition-only model: {floor:.3f} V")
print(f"best test RMSE in D-7: {four['RMSE'].min():.3f} V")

## E. Assignment (attendance)

Sections A to D were worked examples. Now you run the same procedure yourself: choose one of the four
models, or keep a model and switch the target to gravimetric capacity, then train it, evaluate it on the
test rows, and save a results table and a parity plot. The two files you save are what you submit on KLMS.

### E-1. Set your student ID
Your ID is only used to name the files you submit.

In [ ]:
## Replace this with your own student ID; it only names your files.
STUDENT_ID = "demo"

if STUDENT_ID == "demo":
    raise ValueError("Set STUDENT_ID to your own ID before running this cell.")

print("Student ID:", STUDENT_ID)

### E-2. Choose your model and target
`MY_MODEL` is one of `linear`, `random_forest`, `hist_gradient_boosting`, `mlp`. `MY_TARGET` is
`average_voltage` (today's target) or `capacity_grav` (mAh/g). Change at least one of them from the defaults.
If you choose `capacity_grav`, remember its long tail from A-2: a few very large capacities can dominate the
RMSE. The split is the same as in B-2, only the target column changes.

In [ ]:
## MY_MODEL: linear | random_forest | hist_gradient_boosting | mlp
MY_MODEL = "random_forest"
## MY_TARGET: average_voltage | capacity_grav
MY_TARGET = "average_voltage"

MODELS = {"linear", "random_forest", "hist_gradient_boosting", "mlp"}
assert MY_MODEL in MODELS, "unknown MY_MODEL"
assert MY_TARGET in {"average_voltage", "capacity_grav"}, "bad MY_TARGET"
y_my = df[MY_TARGET]
y_my_train, y_my_test = y_my.iloc[train_idx], y_my.iloc[test_idx]
print("model:", MY_MODEL, " target:", MY_TARGET,
      " train rows:", len(y_my_train), " test rows:", len(y_my_test))

### E-3. Train, evaluate and plot
Fit your model on `X_train` / `y_my_train` and evaluate it on both splits. Produce two objects:

1. `my_results` — a DataFrame with exactly these columns: `student_id, model, target, split, MAE, RMSE, R2`,
   one row for `train` and one for `test`.
2. `my_fig` — a Matplotlib figure with the test-set parity plot. Reuse `parity` and pass
   `target=MY_TARGET` with `unit="V"` for the voltage or `unit="mAh/g"` for the capacity.

Write it yourself in the empty cell below. For `mlp`, follow D-1 and D-2 (split first, then fit the scaler
on the fit rows) and reuse `to_tensor`, `make_mlp` and `train_mlp`.

Useful pieces from earlier: the model definitions in C-1, `scores(model, X, y)` from B-4 (a dict with MAE,
RMSE and R2) and `parity` from B-5. `pd.DataFrame([...])` turns a list of two dicts, one per split, into
the table.

In [ ]:
## Write your code here.
## Order: build the model for MY_MODEL -> fit on X_train, y_my_train -> scores for train and test -> my_results -> my_fig

### E-4. Save your submission
Running this cell writes two files and, in Colab, downloads them to your computer (allow the two
downloads if the browser asks). Upload exactly these two files on KLMS (W10 Lab); nothing is filled in by
hand. If E-3 is not finished yet, the cell stops with "Complete E-3 first" and no files are written.

1. `hos5_<id>_results.csv` — your results table
2. `hos5_<id>_parity.png` — your test-set parity plot

In [ ]:
## Stop with a clear message if E-3 has not been completed.
missing = [n for n in ("my_results", "my_fig") if n not in globals()]
if missing:
    raise NameError(f"Complete E-3 first. Not defined yet: {missing}")
columns = ["student_id", "model", "target", "split", "MAE", "RMSE", "R2"]
assert list(my_results.columns) == columns, f"columns must be {columns}"
safe_id = "".join(c for c in STUDENT_ID if c.isalnum() or c in "_-")
stem = OUTPUT / f"hos5_{safe_id}"
paths = [Path(f"{stem}_results.csv"), Path(f"{stem}_parity.png")]
my_results.to_csv(paths[0], index=False)
my_fig.savefig(paths[1], dpi=180, bbox_inches="tight")
for path in paths:
    print("created" if path.stat().st_size > 0 else "MISSING", path)
print(my_results.round(3).to_string(index=False))
try:   ## in Colab, download both files to your computer for the KLMS upload
    from google.colab import files
    files.download(str(paths[0])); files.download(str(paths[1]))
except ImportError:
    print("Not in Colab: download the two files from the outputs folder.")
print("Submit both files to KLMS (W10 Lab).")

## Appendix (optional). Two importances and a noise column
This section is not part of the class time; run it after section C if you are curious.
C-4 is the forest's own importance (mean decrease in impurity, MDI). It is computed on the **training** data while
the trees grow. **Permutation importance** shuffles one column of the **test** set at a time and measures how much
the test RMSE rises: it shows what the model relies on for new data, and it works for any model. Here we add one
column of random numbers, refit the forest and compare the two. The cell takes about 20 seconds.

The red bar is the noise column; if a panel has no red bar, the noise did not make that panel's top 15.
The two printed lines give its rank in each. A column of random numbers offers many split points, and
deep trees use them to fit the training rows a little closer, so MDI gives it credit. Shuffling it on the test set barely changes the error, so its
permutation importance stays near zero. When the question is "what does the model need for new
materials?", rank features by permutation importance on held-out data.

In [ ]:
## One extra column of random numbers: it carries no information on the voltage.
rng = np.random.default_rng(0)
X_train_noise = X_train.assign(random_noise=rng.normal(size=len(X_train)))
X_test_noise = X_test.assign(random_noise=rng.normal(size=len(X_test)))
rf_noise = make_pipeline(SimpleImputer(strategy="median"),
                         RandomForestRegressor(n_estimators=200,
                                               n_jobs=-1, random_state=0))
rf_noise.fit(X_train_noise, y_train)

## MDI comes from training. Permutation shuffles one test column at a time.
mdi = pd.Series(
    rf_noise.named_steps["randomforestregressor"].feature_importances_,
    index=X_train_noise.columns)
perm = permutation_importance(rf_noise, X_test_noise, y_test,
                              scoring="neg_root_mean_squared_error",
                              n_repeats=3, random_state=0)
perm = pd.Series(perm.importances_mean, index=X_train_noise.columns)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
panels = [(mdi, "MDI (training data)"),
          (perm, "Permutation (test data): RMSE rise (V)")]
for ax, (imp, title) in zip(axes, panels):
    top = imp.sort_values(ascending=False).head(15)[::-1]
    colors = ["tab:red" if c == "random_noise" else "tab:blue"
              for c in top.index]
    ax.barh(top.index.str.replace("MagpieData ", ""), top.values,
            color=colors)
    ax.set_title(title)
fig.tight_layout()
fig.savefig(OUTPUT / "appendix_mdi_vs_permutation.png", dpi=150,
            bbox_inches="tight")
plt.show()
for name, imp in [("MDI", mdi), ("permutation", perm)]:
    rank = int(imp.rank(ascending=False, method="min")["random_noise"])
    share = 100 * imp["random_noise"] / imp.max()
    print(f"random_noise by {name}: rank {rank} of {len(imp)}, "
          f"{share:.1f}% of the top feature")

## Data sources

### Materials Project insertion electrodes
`Data/mp_li_electrodes.csv` and `Data/mp_li_electrodes_magpie145.csv.gz`. Materials Project insertion electrodes (working ion Li), accessed 2026-10-22, licensed under CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/). Cite: A. Jain, S. P. Ong, G. Hautier, W. Chen, W. D. Richards, S. Dacek, S. Cholia, D. Gunter, D. Skinner, G. Ceder, K. A. Persson, *APL Materials* 1, 011002 (2013), https://doi.org/10.1063/1.4812323. Changes: rows without average_voltage or formula_discharge removed; 0 < V < 6 kept; duplicate battery_id removed; composition features added with matminer (Ward et al., *Comput. Mater. Sci.* 152, 60 (2018), https://doi.org/10.1016/j.commatsci.2018.05.018) and Magpie properties (Ward et al., *npj Comput. Mater.* 2, 16028 (2016), https://doi.org/10.1038/npjcompumats.2016.28).